# Profile Orderflow Strategy v15

## Purpose

v15 is a research-grade rewrite of v14. It preserves the valid auction-market thesis while correcting the implementation, experiment, execution, and reporting defects identified during review.

### Core thesis

- **Location first:** use prior-day / rolling-24h Volume Profile and a rolling five-completed-day composite.
- **Auction state second:** distinguish overlapping balance, higher-value acceptance, lower-value acceptance, failed lower auction, and failed upper auction.
- **Orderflow timing third:** require evidence from three distinct categories-aggression, effectiveness, and acceptance-rather than correlated 3-of-5 votes.
- **Execution last:** signal on a completed 5-minute bar, fill on the next available trade/quote or next bar open, simulate stop/target touches with intrabar high/low and conservative same-bar ordering.

### Important scope

- `VAL_REJECTION_LONG` is the only active production-research setup by default.
- `VAH_REJECTION_SHORT` and `VAH_RECLAIM_LONG` are diagnostics-only by default.
- Binance `bookTicker` is treated as sampled L1 displayed-liquidity context, **not** footprint or stacked imbalance.
- Exact-price absorption is not claimed unless price-level trade data support the calculation.

### NautilusTrader integration

The notebook initializes the Binance BTCUSDT perpetual test instrument through `TestInstrumentProvider` when NautilusTrader is installed. The research engine below remains explicit and auditable because the custom auction/profile state, candidate funnels, and partial-position logic are domain-specific. The output can subsequently be converted to Nautilus `TradeTick`, `QuoteTick`, and order events for a full `BacktestEngine`/`BacktestNode` replay.


In [34]:
from __future__ import annotations

import io
import json
import math
import tempfile
import warnings
import zipfile
from collections import defaultdict, deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd

try:
    from nautilus_trader.test_kit.providers import TestInstrumentProvider
    NAUTILUS_AVAILABLE = True
    INSTRUMENT = TestInstrumentProvider.btcusdt_perp_binance()
except ImportError:
    NAUTILUS_AVAILABLE = False
    INSTRUMENT = None

warnings.filterwarnings("ignore")

SYMBOL = "BTCUSDT"
TICK_SIZE = 0.1
VALUE_AREA_PCT = 0.70
BAR_MINUTES = 5
BAR_MS = BAR_MINUTES * 60_000
BARS_60M = 60 // BAR_MINUTES        # 12 bars
BARS_180M = 180 // BAR_MINUTES      # 36 bars
BARS_24H = 24 * 60 // BAR_MINUTES   # 288 bars

ACCOUNT_START = 100_000.0
BASE_RISK_LONG = 0.0075
BASE_RISK_SHORT = 0.0050
MAX_LEVERAGE = 3.0
MAX_BTC = 5.0
DAILY_LOSS_LIMIT = 0.02

TAKER_FEE_BPS = 5.0
SLIPPAGE_BPS = 1.0
SPREAD_LIMIT_BPS = 10.0

OUT_DIR = Path("outputs/v15")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"NautilusTrader available: {NAUTILUS_AVAILABLE}")
if INSTRUMENT is not None:
    print(f"Instrument: {INSTRUMENT.id}")


NautilusTrader available: True
Instrument: BTCUSDT-PERP.BINANCE


In [35]:
@dataclass(frozen=True)
class VolumeProfile:
    val: float
    poc: float
    vah: float
    total_volume: float
    price_volume: dict[float, float]

    @property
    def midpoint(self) -> float:
        return (self.val + self.vah) / 2.0

    @property
    def width(self) -> float:
        return max(self.vah - self.val, TICK_SIZE)


@dataclass
class Bar:
    ts_start: int
    ts_end: int
    open: float
    high: float
    low: float
    close: float
    volume: float
    buy_volume: float
    sell_volume: float
    delta: float
    delta_pct: float
    bid_qty_sum: float
    ask_qty_sum: float
    spread_bps_median: float
    volume_by_price: dict[float, float] = field(default_factory=dict)

    @property
    def close_location(self) -> float:
        span = self.high - self.low
        return 0.5 if span <= 0 else (self.close - self.low) / span

    @property
    def body_ratio(self) -> float:
        span = self.high - self.low
        return 0.0 if span <= 0 else abs(self.close - self.open) / span

    @property
    def lower_wick_ratio(self) -> float:
        span = self.high - self.low
        return 0.0 if span <= 0 else (min(self.open, self.close) - self.low) / span

    @property
    def upper_wick_ratio(self) -> float:
        span = self.high - self.low
        return 0.0 if span <= 0 else (self.high - max(self.open, self.close)) / span

    @property
    def l1_log_imbalance(self) -> float:
        # Positive = more displayed bid size; negative = more displayed ask size.
        return math.log((self.bid_qty_sum + 1e-9) / (self.ask_qty_sum + 1e-9))


def _bucket_price(px: float) -> float:
    return round(round(px / TICK_SIZE) * TICK_SIZE, 8)


def compute_volume_profile(price_volume: dict[float, float], value_area_pct: float = VALUE_AREA_PCT) -> Optional[VolumeProfile]:
    if not price_volume:
        return None
    pv = {float(k): float(v) for k, v in price_volume.items() if v > 0}
    if not pv:
        return None
    prices = sorted(pv)
    total = sum(pv.values())
    w = [pv[x] for x in prices]
    avg = np.average(prices, weights=w)
    poc = max(prices, key=lambda p: (pv[p], -abs(p - avg)))
    target = total * value_area_pct
    selected = {poc}
    running = pv[poc]
    idx = prices.index(poc)
    left, right = idx - 1, idx + 1
    while running < target and (left >= 0 or right < len(prices)):
        lv = pv[prices[left]] if left >= 0 else -1.0
        rv = pv[prices[right]] if right < len(prices) else -1.0
        if rv > lv:
            selected.add(prices[right]); running += rv; right += 1
        else:
            selected.add(prices[left]); running += lv; left -= 1
    return VolumeProfile(min(selected), poc, max(selected), total, pv)


def merge_profiles(profiles: Iterable[VolumeProfile]) -> Optional[VolumeProfile]:
    merged: dict[float, float] = defaultdict(float)
    for prof in profiles:
        for px, qty in prof.price_volume.items():
            merged[px] += qty
    return compute_volume_profile(dict(merged))


def profile_from_bars(bars: list[Bar]) -> Optional[VolumeProfile]:
    merged: dict[float, float] = defaultdict(float)
    for bar in bars:
        for px, qty in bar.volume_by_price.items():
            merged[px] += qty
    return compute_volume_profile(dict(merged))


In [36]:
def build_5m_bars(agg_trades: pd.DataFrame, book_ticker: Optional[pd.DataFrame] = None) -> list[Bar]:
    """Build completed 5-minute bars without future leakage.
    
    Required aggTrades columns: transact_time, price, quantity, is_buyer_maker.
    `is_buyer_maker=True` means the buyer was passive, so the aggressive side was sell.
    """
    required = {"transact_time", "price", "quantity", "is_buyer_maker"}
    missing = required - set(agg_trades.columns)
    if missing:
        raise ValueError(f"Missing aggTrades columns: {sorted(missing)}")
    
    t = agg_trades.copy()
    t = t.sort_values("transact_time", kind="stable")
    t["bar_start"] = (t["transact_time"].astype("int64") // BAR_MS) * BAR_MS
    t["aggr_buy_qty"] = np.where(~t["is_buyer_maker"].astype(bool), t["quantity"], 0.0)
    t["aggr_sell_qty"] = np.where(t["is_buyer_maker"].astype(bool), t["quantity"], 0.0)
    t["bucket"] = t["price"].map(_bucket_price)
    
    quote_groups: dict[int, tuple[float, float, float]] = {}
    if book_ticker is not None and not book_ticker.empty:
        q = book_ticker.copy().sort_values("transact_time", kind="stable")
        q["bar_start"] = (q["transact_time"].astype("int64") // BAR_MS) * BAR_MS
        q["spread_bps"] = (q["best_ask_price"] - q["best_bid_price"]) / ((q["best_ask_price"] + q["best_bid_price"]) / 2) * 10_000
        for ts, g in q.groupby("bar_start", sort=True):
            quote_groups[int(ts)] = (
                float(g["best_bid_qty"].sum()),
                float(g["best_ask_qty"].sum()),
                float(g["spread_bps"].median()),
            )
    
    out: list[Bar] = []
    for ts, g in t.groupby("bar_start", sort=True):
        buy = float(g["aggr_buy_qty"].sum())
        sell = float(g["aggr_sell_qty"].sum())
        vol = buy + sell
        pv = g.groupby("bucket", sort=False)["quantity"].sum().astype(float).to_dict()
        bid_sum, ask_sum, spread = quote_groups.get(int(ts), (0.0, 0.0, np.nan))
        out.append(Bar(
            ts_start=int(ts), ts_end=int(ts) + BAR_MS,
            open=float(g.iloc[0]["price"]), high=float(g["price"].max()),
            low=float(g["price"].min()), close=float(g.iloc[-1]["price"]),
            volume=vol, buy_volume=buy, sell_volume=sell,
            delta=buy - sell, delta_pct=0.0 if vol <= 0 else (buy - sell) / vol,
            bid_qty_sum=bid_sum, ask_qty_sum=ask_sum,
            spread_bps_median=spread, volume_by_price=pv,
        ))
    return out


def assert_bars_valid(bars: list[Bar]) -> None:
    last = -1
    for b in bars:
        assert b.ts_start > last, "Bars must be strictly timestamp ordered"
        assert b.low <= min(b.open, b.close) <= max(b.open, b.close) <= b.high
        assert abs((b.buy_volume + b.sell_volume) - b.volume) < 1e-6
        assert -1.0 <= b.delta_pct <= 1.0
        last = b.ts_start


## Auction context model

v15 removes the six-vote composite score. Context is represented as interpretable geometry and acceptance:

- **Composite geometry:** rolling five completed days only; no current-day or future data.
- **Intraday geometry:** true rolling 60-minute (12 bars) and 180-minute (36 bars) profiles.
- **Material migration:** profile shifts are normalized by active value-area width and ATR, and must persist.
- **Acceptance:** uses both close counts and the fraction of volume transacted outside the reference boundary.

CVD is deliberately excluded from the higher-timeframe context and reserved for entry orderflow timing.


In [37]:
@dataclass(frozen=True)
class ContextSnapshot:
    composite_geometry: str
    migration: str
    acceptance: str
    p60: Optional[VolumeProfile]
    p180: Optional[VolumeProfile]
    poc_shift_60_norm: float
    midpoint_shift_60_norm: float
    overlap_60: float


def value_overlap(a: VolumeProfile, b: VolumeProfile) -> float:
    overlap = max(0.0, min(a.vah, b.vah) - max(a.val, b.val))
    denom = max(min(a.width, b.width), TICK_SIZE)
    return min(overlap / denom, 1.0)


class AuctionContextEngine:
    def __init__(self, persistence: int = 2):
        self.bars: deque[Bar] = deque(maxlen=BARS_24H)
        self._p60_bars: deque[Bar] = deque(maxlen=BARS_60M)
        self._p180_bars: deque[Bar] = deque(maxlen=BARS_180M)
        self._p60_merged: dict[float, float] = {}
        self._p180_merged: dict[float, float] = {}
        self.prev_p60: Optional[VolumeProfile] = None
        self.prev_p180: Optional[VolumeProfile] = None
        self.persistence = persistence
        self.raw_migration_history: deque[str] = deque(maxlen=persistence)

    def _update_rolling_profile(self, bar: Bar, window: int, bars_deque: deque, merged: dict[float, float]) -> Optional[VolumeProfile]:
        if len(bars_deque) == window:
            oldest = bars_deque.popleft()
            for px, vol in oldest.volume_by_price.items():
                new_val = merged.get(px, 0.0) - vol
                if new_val <= 0.0:
                    merged.pop(px, None)
                else:
                    merged[px] = new_val
        bars_deque.append(bar)
        for px, vol in bar.volume_by_price.items():
            merged[px] = merged.get(px, 0.0) + vol
        if len(bars_deque) < window:
            return None
        return compute_volume_profile(dict(merged), VALUE_AREA_PCT) if merged else None

    def update(self, bar: Bar, active: VolumeProfile, composite: VolumeProfile, atr: float) -> ContextSnapshot:
        self.bars.append(bar)
        bars = list(self.bars)
        p60 = self._update_rolling_profile(bar, BARS_60M, self._p60_bars, self._p60_merged) if len(bars) >= 1 else None
        p180 = self._update_rolling_profile(bar, BARS_180M, self._p180_bars, self._p180_merged) if len(bars) >= 1 else None

        geometry = "OVERLAPPING_COMPOSITE"
        if active.val > composite.vah:
            geometry = "HIGHER_VALUE"
        elif active.vah < composite.val:
            geometry = "LOWER_VALUE"
        elif value_overlap(active, composite) < 0.30:
            geometry = "DISPLACED_VALUE"

        poc_shift = midpoint_shift = 0.0
        overlap = 1.0
        raw = "NO_CLEAR"
        norm = max(active.width, atr * 2.0, TICK_SIZE)
        if p60 is not None and self.prev_p60 is not None:
            poc_shift = (p60.poc - self.prev_p60.poc) / norm
            midpoint_shift = (p60.midpoint - self.prev_p60.midpoint) / norm
            overlap = value_overlap(p60, self.prev_p60)
            material_up = poc_shift >= 0.05 and midpoint_shift >= 0.03
            material_dn = poc_shift <= -0.05 and midpoint_shift <= -0.03
            if overlap >= 0.75 and abs(poc_shift) < 0.05 and abs(midpoint_shift) < 0.03:
                raw = "OVERLAPPING_VALUE"
            elif material_up:
                raw = "HIGHER_VALUE"
            elif material_dn:
                raw = "LOWER_VALUE"

        self.raw_migration_history.append(raw)
        migration = raw if len(self.raw_migration_history) == self.persistence and len(set(self.raw_migration_history)) == 1 else "TRANSITION"

        recent = bars[-6:]
        vol_total = sum(x.volume for x in recent) or 1.0
        vol_below_val = sum(sum(q for px, q in x.volume_by_price.items() if px < active.val) for x in recent) / vol_total
        vol_above_vah = sum(sum(q for px, q in x.volume_by_price.items() if px > active.vah) for x in recent) / vol_total
        closes_below = sum(x.close < active.val for x in recent)
        closes_above = sum(x.close > active.vah for x in recent)
        if closes_below >= 3 and vol_below_val >= 0.30:
            acceptance = "ACCEPTED_BELOW_VALUE"
        elif closes_above >= 3 and vol_above_vah >= 0.30:
            acceptance = "ACCEPTED_ABOVE_VALUE"
        elif bar.close >= active.val and any(x.low < active.val for x in recent):
            acceptance = "RETURNED_FROM_BELOW"
        elif bar.close <= active.vah and any(x.high > active.vah for x in recent):
            acceptance = "RETURNED_FROM_ABOVE"
        else:
            acceptance = "NO_ACCEPTANCE"

        snap = ContextSnapshot(geometry, migration, acceptance, p60, p180, poc_shift, midpoint_shift, overlap)
        self.prev_p60, self.prev_p180 = p60, p180
        return snap


## Failed-auction signal model

A rejection is an event sequence, not merely a candle shape.

### VAL failed-auction long

1. The market probes below VAL within a bounded depth.
2. The completed 5-minute signal bar closes back above VAL, in the upper portion of its range.
3. Lower-value acceptance is absent.
4. At least one **aggression-failure** signal is present: lower-low/CVD non-confirmation or large negative delta with poor downside efficiency.
5. At least one **buyer-response** signal is present: positive delta percentile, rising CVD after the probe, or persistent positive displayed-bid context.
6. Participation is adequate.
7. Entry occurs on the next bar only if the reclaim remains valid.

The short setup uses the symmetric upper-auction logic but remains diagnostics-only by default.


In [38]:
@dataclass(frozen=True)
class Signal:
    signal_id: str
    setup: str
    side: str
    signal_ts: int
    level: float
    signal_high: float
    signal_low: float
    stop_anchor: float
    context: ContextSnapshot
    diagnostics: dict[str, Any]


class OrderflowEngine:
    def __init__(self):
        self.cvd = 0.0
        self.cvd_hist: deque[float] = deque(maxlen=120)
        self.delta_pct_hist: deque[float] = deque(maxlen=120)
        self.volume_hist: deque[float] = deque(maxlen=120)
        self.close_hist: deque[float] = deque(maxlen=120)
        self.low_hist: deque[float] = deque(maxlen=120)
        self.high_hist: deque[float] = deque(maxlen=120)

    def update(self, bar: Bar) -> None:
        self.cvd += bar.delta
        self.cvd_hist.append(self.cvd)
        self.delta_pct_hist.append(bar.delta_pct)
        self.volume_hist.append(bar.volume)
        self.close_hist.append(bar.close)
        self.low_hist.append(bar.low)
        self.high_hist.append(bar.high)

    def percentile(self, value: float, values: deque[float], n: int = 60) -> float:
        arr = np.asarray(list(values)[-n:], dtype=float)
        if len(arr) < 10:
            return 0.5
        return float(np.mean(arr <= value))

    def bullish_lower_low_divergence(self) -> bool:
        if len(self.low_hist) < 12 or len(self.cvd_hist) < 12:
            return False
        old_low = min(list(self.low_hist)[-12:-6])
        new_low = min(list(self.low_hist)[-6:])
        old_cvd_low = min(list(self.cvd_hist)[-12:-6])
        new_cvd_low = min(list(self.cvd_hist)[-6:])
        return new_low < old_low and new_cvd_low >= old_cvd_low

    def bearish_higher_high_divergence(self) -> bool:
        if len(self.high_hist) < 12 or len(self.cvd_hist) < 12:
            return False
        old_high = max(list(self.high_hist)[-12:-6])
        new_high = max(list(self.high_hist)[-6:])
        old_cvd_high = max(list(self.cvd_hist)[-12:-6])
        new_cvd_high = max(list(self.cvd_hist)[-6:])
        return new_high > old_high and new_cvd_high <= old_cvd_high

    def cvd_recovery(self, bullish: bool) -> bool:
        if len(self.cvd_hist) < 6:
            return False
        change = self.cvd_hist[-1] - self.cvd_hist[-4]
        return change > 0 if bullish else change < 0

    def participation_ok(self, bar: Bar) -> bool:
        if len(self.volume_hist) < 20:
            return False
        prior = np.asarray(list(self.volume_hist)[-20:-1], dtype=float)
        return len(prior) >= 10 and bar.volume >= np.median(prior)

    def val_failed_auction(self, bar: Bar, active: VolumeProfile, context: ContextSnapshot, atr: float) -> tuple[bool, dict[str, Any]]:
        probe_depth = max(active.val - bar.low, 0.0)
        max_probe = max(0.50 * atr, 0.20 * active.width)
        bounded_probe = bar.low <= active.val and probe_depth <= max_probe
        reclaim = active.val <= bar.close <= active.vah
        shape = bar.close_location >= 0.55 and bar.lower_wick_ratio >= 0.20
        no_lower_acceptance = context.acceptance != "ACCEPTED_BELOW_VALUE" and context.migration != "LOWER_VALUE"

        delta_pct_rank = self.percentile(bar.delta_pct, self.delta_pct_hist)
        negative_effort_failure = bar.delta_pct <= -0.10 and (bar.close - bar.low) >= 0.50 * max(bar.high - bar.low, TICK_SIZE)
        aggression_failure = self.bullish_lower_low_divergence() or negative_effort_failure

        l1_bid_support = bar.l1_log_imbalance >= math.log(1.20)
        buyer_response = delta_pct_rank >= 0.65 or self.cvd_recovery(True) or l1_bid_support
        participation = self.participation_ok(bar)
        spread_ok = np.isnan(bar.spread_bps_median) or bar.spread_bps_median <= SPREAD_LIMIT_BPS

        details = {
            "bounded_probe": bounded_probe, "probe_depth": probe_depth,
            "reclaim": reclaim, "shape": shape,
            "no_lower_acceptance": no_lower_acceptance,
            "aggression_failure": aggression_failure,
            "buyer_response": buyer_response, "participation": participation,
            "spread_ok": spread_ok, "delta_pct_rank": delta_pct_rank,
            "l1_bid_support": l1_bid_support,
        }
        return all([bounded_probe, reclaim, shape, no_lower_acceptance, aggression_failure, buyer_response, participation, spread_ok]), details

    def vah_failed_auction(self, bar: Bar, active: VolumeProfile, context: ContextSnapshot, atr: float) -> tuple[bool, dict[str, Any]]:
        probe_depth = max(bar.high - active.vah, 0.0)
        max_probe = max(0.50 * atr, 0.20 * active.width)
        bounded_probe = bar.high >= active.vah and probe_depth <= max_probe
        reclaim = active.val <= bar.close <= active.vah
        shape = bar.close_location <= 0.45 and bar.upper_wick_ratio >= 0.20
        no_higher_acceptance = context.acceptance != "ACCEPTED_ABOVE_VALUE" and context.migration != "HIGHER_VALUE"

        delta_pct_rank = self.percentile(bar.delta_pct, self.delta_pct_hist)
        positive_effort_failure = bar.delta_pct >= 0.10 and (bar.high - bar.close) >= 0.50 * max(bar.high - bar.low, TICK_SIZE)
        aggression_failure = self.bearish_higher_high_divergence() or positive_effort_failure
        l1_ask_support = bar.l1_log_imbalance <= -math.log(1.20)
        seller_response = delta_pct_rank <= 0.35 or self.cvd_recovery(False) or l1_ask_support
        participation = self.participation_ok(bar)
        spread_ok = np.isnan(bar.spread_bps_median) or bar.spread_bps_median <= SPREAD_LIMIT_BPS
        details = {
            "bounded_probe": bounded_probe, "probe_depth": probe_depth,
            "reclaim": reclaim, "shape": shape,
            "no_higher_acceptance": no_higher_acceptance,
            "aggression_failure": aggression_failure,
            "seller_response": seller_response, "participation": participation,
            "spread_ok": spread_ok, "delta_pct_rank": delta_pct_rank,
            "l1_ask_support": l1_ask_support,
        }
        return all([bounded_probe, reclaim, shape, no_higher_acceptance, aggression_failure, seller_response, participation, spread_ok]), details


In [39]:
@dataclass
class PendingEntry:
    signal: Signal
    expires_after_ts: int


@dataclass
class Position:
    position_id: str
    setup: str
    side: str
    entry_ts: int
    entry_price: float
    initial_size: float
    remaining_size: float
    stop_price: float
    initial_r: float
    t1: float
    t2: float
    entry_context: ContextSnapshot
    t1_done: bool = False
    t2_done: bool = False
    runner_active: bool = False
    trail_extreme: float = 0.0
    realized_pnl: float = 0.0
    fees: float = 0.0
    mfe: float = 0.0
    mae: float = 0.0


@dataclass(frozen=True)
class ExitLeg:
    position_id: str
    exit_leg_id: str
    setup: str
    side: str
    entry_ts: int
    exit_ts: int
    entry_price: float
    exit_price: float
    size: float
    reason: str
    gross_pnl: float
    fees: float
    net_pnl: float


class RiskEngine:
    @staticmethod
    def size(entry: float, stop: float, risk_fraction: float, equity: float) -> float:
        distance = abs(entry - stop)
        if distance <= 0:
            return 0.0
        raw = equity * risk_fraction / distance
        cap_leverage = equity * MAX_LEVERAGE / entry
        return max(0.0, min(raw, cap_leverage, MAX_BTC))


class ExecutionEngine:
    """Auditable position engine with conservative same-bar ordering.
    
    Assumption: when stop and target are both touched in the same 5M bar and no tick replay
    is supplied, stop is processed first. This is deliberately conservative.
    """
    def __init__(self, equity: float = ACCOUNT_START):
        self.equity = equity
        self.position: Optional[Position] = None
        self.legs: list[ExitLeg] = []
        self.positions_closed: list[dict[str, Any]] = []
        self._leg_counter = 0

    @staticmethod
    def _adverse_fill(price: float, side: str, entering: bool) -> float:
        bps = SLIPPAGE_BPS / 10_000
        buy = (side == "LONG" and entering) or (side == "SHORT" and not entering)
        return price * (1 + bps if buy else 1 - bps)

    @staticmethod
    def _gross(side: str, entry: float, exit_: float, size: float) -> float:
        return (exit_ - entry) * size if side == "LONG" else (entry - exit_) * size

    def open(self, signal: Signal, bar: Bar, active: VolumeProfile, atr: float, risk_fraction: float) -> bool:
        if self.position is not None:
            return False
        # Signal was created on prior completed bar; fill at this bar open with adverse slippage.
        entry = self._adverse_fill(bar.open, signal.side, True)
        if signal.side == "LONG":
            if bar.open < active.val:  # reclaim failed before entry
                return False
            stop_anchor = min(signal.stop_anchor, signal.signal_low, active.val)
            stop = stop_anchor - max(0.10 * atr, entry * 0.0005)
            r = entry - stop
            t1, t2 = active.poc, active.vah
            if not (stop < entry < t1 < t2):
                return False
            risk_fraction = risk_fraction if signal.context.migration != "LOWER_VALUE" else 0.0
        else:
            if bar.open > active.vah:
                return False
            stop_anchor = max(signal.stop_anchor, signal.signal_high, active.vah)
            stop = stop_anchor + max(0.10 * atr, entry * 0.0005)
            r = stop - entry
            t1, t2 = active.poc, active.val
            if not (t2 < t1 < entry < stop):
                return False
            risk_fraction = risk_fraction if signal.context.migration != "HIGHER_VALUE" else 0.0
        if risk_fraction <= 0 or abs(t1 - entry) < 0.8 * r:
            return False
        size = RiskEngine.size(entry, stop, risk_fraction, self.equity)
        if size < 0.001:
            return False
        pid = f"{signal.setup}-{signal.signal_ts}"
        self.position = Position(pid, signal.setup, signal.side, bar.ts_start, entry, size, size,
                                 stop, r, t1, t2, signal.context, trail_extreme=entry)
        return True

    def _close(self, price: float, ts: int, requested_size: float, reason: str) -> None:
        p = self.position
        if p is None:
            return
        size = min(max(requested_size, 0.0), p.remaining_size)
        if size <= 1e-12:
            return
        exit_price = self._adverse_fill(price, p.side, False)
        gross = self._gross(p.side, p.entry_price, exit_price, size)
        fee = (p.entry_price * size + exit_price * size) * TAKER_FEE_BPS / 10_000
        net = gross - fee
        p.realized_pnl += net
        p.fees += fee
        p.remaining_size -= size
        self.equity += net
        self._leg_counter += 1
        self.legs.append(ExitLeg(p.position_id, f"{p.position_id}-L{self._leg_counter}", p.setup, p.side,
                                 p.entry_ts, ts, p.entry_price, exit_price, size, reason, gross, fee,
                                 net))
        assert p.remaining_size >= -1e-9
        closed = p.initial_size - p.remaining_size
        assert closed <= p.initial_size + 1e-9
        if p.remaining_size <= 1e-9:
            self.positions_closed.append({
                "position_id": p.position_id, "setup": p.setup, "side": p.side,
                "entry_ts": p.entry_ts, "net_pnl": p.realized_pnl, "fees": p.fees,
                "mfe_R": p.mfe / p.initial_r, "mae_R": p.mae / p.initial_r,
                "entry_context": asdict(p.entry_context),
            })
            self.position = None

    def on_bar(self, bar: Bar, current_context: ContextSnapshot, active: VolumeProfile, atr: float) -> None:
        p = self.position
        if p is None:
            return
        if p.side == "LONG":
            p.mfe = max(p.mfe, bar.high - p.entry_price)
            p.mae = min(p.mae, bar.low - p.entry_price)
            stop_hit = bar.low <= p.stop_price
            t1_hit = (not p.t1_done) and bar.high >= p.t1
            t2_hit = p.t1_done and (not p.t2_done) and bar.high >= p.t2
        else:
            p.mfe = max(p.mfe, p.entry_price - bar.low)
            p.mae = min(p.mae, p.entry_price - bar.high)
            stop_hit = bar.high >= p.stop_price
            t1_hit = (not p.t1_done) and bar.low <= p.t1
            t2_hit = p.t1_done and (not p.t2_done) and bar.low <= p.t2

        # Conservative sequence when OHLC cannot resolve path.
        if stop_hit:
            self._close(p.stop_price, bar.ts_start, p.remaining_size, "STOP")
            return

        if t1_hit and self.position is not None:
            self._close(p.t1, bar.ts_start, p.initial_size * 0.40, "T1_POC")
            if self.position is None:
                return
            p.t1_done = True
            p.stop_price = max(p.stop_price, p.entry_price) if p.side == "LONG" else min(p.stop_price, p.entry_price)

        extension_ok = (
            (p.side == "LONG" and current_context.migration != "LOWER_VALUE" and current_context.acceptance != "ACCEPTED_BELOW_VALUE") or
            (p.side == "SHORT" and current_context.migration != "HIGHER_VALUE" and current_context.acceptance != "ACCEPTED_ABOVE_VALUE")
        )
        if t2_hit and extension_ok and self.position is not None:
            self._close(p.t2, bar.ts_start, p.initial_size * 0.40, "T2_OPPOSITE_VA")
            if self.position is None:
                return
            p.t2_done = True
            p.runner_active = True

        if self.position is not None and p.runner_active:
            if p.side == "LONG":
                p.trail_extreme = max(p.trail_extreme, bar.high)
                p.stop_price = max(p.stop_price, p.trail_extreme - max(0.75 * p.initial_r, atr))
            else:
                p.trail_extreme = min(p.trail_extreme, bar.low)
                p.stop_price = min(p.stop_price, p.trail_extreme + max(0.75 * p.initial_r, atr))


In [40]:
class StrategyV15:
    def __init__(self, active_profile: VolumeProfile, composite_profile: VolumeProfile, enable_short: bool = False, enable_reclaim_trading: bool = False):
        self.active = active_profile
        self.composite = composite_profile
        self.enable_short = enable_short
        self.enable_reclaim_trading = enable_reclaim_trading
        self.context_engine = AuctionContextEngine()
        self.orderflow = OrderflowEngine()
        self.execution = ExecutionEngine()
        self.pending: Optional[PendingEntry] = None
        self.bars: deque[Bar] = deque(maxlen=BARS_24H)
        self.ranges: deque[float] = deque(maxlen=14)
        self.candidate_log: list[dict[str, Any]] = []
        self.reclaim_log: list[dict[str, Any]] = []
        self.last_context: Optional[ContextSnapshot] = None
        self.entries_by_setup: dict[str, int] = defaultdict(int)

    @property
    def atr(self) -> float:
        return float(np.mean(self.ranges)) if len(self.ranges) >= 14 else 0.0

    def _new_signal(self, setup: str, side: str, bar: Bar, level: float,
                    context: ContextSnapshot, details: dict[str, Any]) -> Signal:
        return Signal(f"{setup}-{bar.ts_start}", setup, side, bar.ts_start, level,
                      bar.high, bar.low, bar.low if side == "LONG" else bar.high,
                      context, details)

    def _record_reclaim_diagnostic(self, bar: Bar, context: ContextSnapshot) -> None:
        recent = list(self.bars)
        crossed = len(recent) >= 2 and recent[-2].close <= self.active.vah < bar.close
        hold_3 = len(recent) >= 3 and all(x.close > self.active.vah for x in recent[-3:])
        high_volume = self.orderflow.participation_ok(bar)
        strong_close = bar.close_location >= 0.75 and bar.body_ratio >= 0.50
        distributed = max(bar.volume_by_price.values(), default=0.0) / max(bar.volume, 1e-9) <= 0.35
        self.reclaim_log.append({
            "ts": bar.ts_start, "crossed_above_vah": crossed, "hold_3": hold_3,
            "migration": context.migration, "acceptance": context.acceptance,
            "high_volume": high_volume, "delta_pct": bar.delta_pct,
            "strong_close": strong_close, "distributed_volume": distributed,
        })

    def on_bar(self, bar: Bar) -> None:
        self.bars.append(bar)
        self.ranges.append(bar.high - bar.low)
        self.orderflow.update(bar)
        if self.atr <= 0:
            return

        # Context updates on every completed bar, including while holding a position.
        context = self.context_engine.update(bar, self.active, self.composite, self.atr)
        self.last_context = context
        self._record_reclaim_diagnostic(bar, context)

        # Existing position is managed before evaluating new signals.
        if self.execution.position is not None:
            self.execution.on_bar(bar, context, self.active, self.atr)
            return

        # Pending signals fill only on the next bar.
        if self.pending is not None:
            if bar.ts_start <= self.pending.expires_after_ts:
                risk = BASE_RISK_LONG if self.pending.signal.side == "LONG" else BASE_RISK_SHORT
                opened = self.execution.open(self.pending.signal, bar, self.active, self.atr, risk)
                self.candidate_log.append({"signal_id": self.pending.signal.signal_id, "stage": "NEXT_BAR_ENTRY", "passed": opened})
            self.pending = None
            if self.execution.position is not None:
                return

        long_ok, long_details = self.orderflow.val_failed_auction(bar, self.active, context, self.atr)
        self.candidate_log.append({"ts": bar.ts_start, "setup": "VAL_REJECTION_LONG", **long_details, "qualified": long_ok})
        if long_ok and self.entries_by_setup["VAL_REJECTION_LONG"] < 1:
            signal = self._new_signal("VAL_REJECTION_LONG", "LONG", bar, self.active.val, context, long_details)
            self.pending = PendingEntry(signal, bar.ts_start + BAR_MS)
            self.entries_by_setup["VAL_REJECTION_LONG"] += 1
            return

        short_ok, short_details = self.orderflow.vah_failed_auction(bar, self.active, context, self.atr)
        self.candidate_log.append({"ts": bar.ts_start, "setup": "VAH_REJECTION_SHORT", **short_details, "qualified": short_ok})
        if self.enable_short and short_ok and self.entries_by_setup["VAH_REJECTION_SHORT"] < 1:
            signal = self._new_signal("VAH_REJECTION_SHORT", "SHORT", bar, self.active.vah, context, short_details)
            self.pending = PendingEntry(signal, bar.ts_start + BAR_MS)
            self.entries_by_setup["VAH_REJECTION_SHORT"] += 1

    def force_close(self, final_bar: Bar) -> None:
        if self.execution.position is not None:
            self.execution._close(final_bar.close, final_bar.ts_start,
                                  self.execution.position.remaining_size, "EOD")

    def export(self, prefix: str = "v15") -> None:
        pd.DataFrame([asdict(x) for x in self.execution.legs]).to_csv(OUT_DIR / f"{prefix}_exit_legs.csv", index=False)
        pd.DataFrame(self.execution.positions_closed).to_csv(OUT_DIR / f"{prefix}_positions.csv", index=False)
        pd.DataFrame(self.candidate_log).to_csv(OUT_DIR / f"{prefix}_candidate_funnel.csv", index=False)
        pd.DataFrame(self.reclaim_log).to_csv(OUT_DIR / f"{prefix}_vah_reclaim_diagnostic.csv", index=False)


## Rolling profile protocol and validation

For each test day:

1. Build the active profile exclusively from data available before the day begins.
2. Build the composite from the previous five completed UTC days, recalculated daily.
3. Warm the 60-minute and 180-minute buffers with prior bars so context is available near the day open.
4. Freeze the active profile for the day unless a separately tested truly rolling mode is selected.
5. Keep the raw timestamp audit: every profile constituent must satisfy `trade_timestamp` < `decision_timestamp`.

### Required output metrics

Report independent counts for signals, pending entries, opened positions, and exit legs. Win rate, MFE, MAE, profit factor, and context attribution must be position-level. Also report full sample, best-trade-excluded, best-day-excluded, leave-one-day-out, fees, slippage, and maximum drawdown.


In [41]:
def summarize_positions(positions: pd.DataFrame) -> dict[str, float]:
    if positions.empty:
        return {"positions": 0, "net_pnl": 0.0, "win_rate": np.nan, "profit_factor": np.nan}
    pnl = positions["net_pnl"].astype(float)
    gross_win = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return {
        "positions": int(len(positions)),
        "net_pnl": float(pnl.sum()),
        "win_rate": float((pnl > 0).mean()),
        "profit_factor": float(gross_win / gross_loss) if gross_loss > 0 else np.inf,
        "median_trade": float(pnl.median()),
        "best_trade_excluded": float(pnl.sum() - pnl.max()),
    }


def validate_execution_invariants(engine: ExecutionEngine) -> None:
    legs = pd.DataFrame([asdict(x) for x in engine.legs])
    if legs.empty:
        return
    grouped = legs.groupby("position_id")["size"].sum()
    for rec in engine.positions_closed:
        pid = rec["position_id"]
        assert pid in grouped.index
        assert np.isfinite(legs[legs["position_id"] == pid][["entry_price", "exit_price", "size", "gross_pnl", "fees", "net_pnl"]].to_numpy()).all()
        assert (legs["size"] > 0).all()
        assert np.allclose(legs["gross_pnl"] - legs["fees"], legs["net_pnl"])


def synthetic_smoke_test() -> None:
    # Profile test
    pv = {99.0: 10, 100.0: 40, 101.0: 30, 102.0: 20}
    prof = compute_volume_profile(pv)
    assert prof is not None and prof.poc == 100.0 and prof.val <= prof.poc <= prof.vah

    # Position accounting test: T1 and T2 can each execute once; total closed <= initial.
    ctx = ContextSnapshot("OVERLAPPING_COMPOSITE", "OVERLAPPING_VALUE", "NO_ACCEPTANCE", prof, prof, 0, 0, 1)
    sig = Signal("TEST", "VAL_REJECTION_LONG", "LONG", 0, 99, 101, 99, 99, ctx, {})
    active = VolumeProfile(99, 101, 103, 100, pv)
    eng = ExecutionEngine(100_000)
    open_bar = Bar(BAR_MS, 2*BAR_MS, 100, 100.5, 99.8, 100.2, 100, 60, 40, 20, .2, 100, 80, 1, pv)
    assert eng.open(sig, open_bar, active, atr=1.0, risk_fraction=0.001)
    p = eng.position
    assert p is not None
    b1 = Bar(2*BAR_MS, 3*BAR_MS, 100.2, 101.2, 100.1, 101, 100, 60, 40, 20, .2, 100, 80, 1, pv)
    eng.on_bar(b1, ctx, active, atr=1.0)
    b2 = Bar(3*BAR_MS, 4*BAR_MS, 101, 103.2, 100.9, 103, 100, 60, 40, 20, .2, 100, 80, 1, pv)
    eng.on_bar(b2, ctx, active, atr=1.0)
    b3 = Bar(4*BAR_MS, 5*BAR_MS, 103, 103.4, 102.8, 103.1, 100, 60, 40, 20, .2, 100, 80, 1, pv)
    eng.on_bar(b3, ctx, active, atr=1.0)
    reasons = [x.reason for x in eng.legs]
    assert reasons.count("T1_POC") <= 1
    assert reasons.count("T2_OPPOSITE_VA") <= 1
    if eng.position is not None:
        eng._close(103.1, b3.ts_start, eng.position.remaining_size, "TEST_END")
    validate_execution_invariants(eng)


synthetic_smoke_test()
print("Synthetic smoke tests passed")


Synthetic smoke tests passed


In [42]:
def read_daily_aggtrades(path: str | Path) -> pd.DataFrame:
    return pd.read_parquet(path)



def read_daily_bookticker(path: str | Path) -> pd.DataFrame:
    return pd.DataFrame()



def profile_for_date(date: str, agg_paths: dict[str, str | Path]) -> VolumeProfile:
    trades = read_daily_aggtrades(agg_paths[date])
    bars = build_5m_bars(trades)
    prof = profile_from_bars(bars)
    if prof is None:
        raise ValueError(f"No profile could be built for {date}")
    return prof


def rolling_completed_day_composite(date_index: int, dates: list[str], daily_profiles: dict[str, VolumeProfile], n: int = 5) -> VolumeProfile:
    prior_dates = dates[max(0, date_index - n):date_index]
    if len(prior_dates) < n:
        raise ValueError(f"Need {n} completed days before {dates[date_index]}; found {len(prior_dates)}")
    comp = merge_profiles([daily_profiles[d] for d in prior_dates])
    if comp is None:
        raise ValueError("Composite profile is empty")
    return comp


def run_v15_backtest(
    all_dates: list[str],
    test_dates: list[str],
    agg_paths: dict[str, str | Path],
    book_paths: Optional[dict[str, str | Path]] = None,
    enable_short: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run one independent StrategyV15 instance per UTC day.
    
    all_dates must be chronologically ordered and include at least five completed dates
    before the first test date. Active profile is the immediately prior completed day;
    composite is the previous five completed days. The previous 180 minutes are used only
    to warm context/orderflow state and are never traded.
    """
    all_dates = sorted(all_dates)
    missing = [d for d in all_dates if d not in agg_paths]
    if missing:
        raise ValueError(f"Missing aggTrades paths for: {missing}")

    daily_bars: dict[str, list[Bar]] = {}
    daily_profiles: dict[str, VolumeProfile] = {}
    for idx2, d in enumerate(all_dates, 1):
        trades = read_daily_aggtrades(agg_paths[d])
        quotes = None
        if book_paths and d in book_paths:
            quotes = read_daily_bookticker(book_paths[d])
        bars = build_5m_bars(trades, quotes)
        assert_bars_valid(bars)
        daily_bars[d] = bars
        prof = profile_from_bars(bars)
        if prof is None:
            raise ValueError(f"Empty daily profile: {d}")
        daily_profiles[d] = prof
        print(f"{len(bars)} bars, profile val={prof.val:.0f}")

    all_legs, all_positions, all_candidates, all_reclaims = [], [], [], []
    equity = ACCOUNT_START
    for idx2, d in enumerate(test_dates, 1):
        idx = all_dates.index(d)
        if idx < 5:
            raise ValueError(f"Insufficient history before {d}")
        active = daily_profiles[all_dates[idx - 1]]
        composite = rolling_completed_day_composite(idx, all_dates, daily_profiles, n=5)
        strat = StrategyV15(active, composite, enable_short=enable_short)
        strat.execution.equity = equity

        # Warm state with exactly the previous 180 minutes, but do not generate entries.
        warm = daily_bars[all_dates[idx - 1]][-BARS_180M:]
        for b in warm:
            strat.bars.append(b)
            strat.ranges.append(b.high - b.low)
            strat.orderflow.update(b)
            if strat.atr > 0:
                strat.last_context = strat.context_engine.update(b, active, composite, strat.atr)
        print(f"    [{idx2}/{len(test_dates)}] {d}: warm... running {len(daily_bars[d])} bars...", end=" ", flush=True)
        strat.entries_by_setup.clear()
        strat.pending = None

        for b in daily_bars[d]:
            strat.on_bar(b)
        if daily_bars[d]:
            strat.force_close(daily_bars[d][-1])
        validate_execution_invariants(strat.execution)
        print(f"pos={len(strat.execution.positions_closed)}, equity={equity:.0f}")
        equity = strat.execution.equity

        for x in strat.execution.legs:
            rec = asdict(x); rec["day"] = d; all_legs.append(rec)
        for rec in strat.execution.positions_closed:
            out = dict(rec); out["day"] = d; all_positions.append(out)
        for rec in strat.candidate_log:
            out = dict(rec); out["day"] = d; all_candidates.append(out)
        for rec in strat.reclaim_log:
            out = dict(rec); out["day"] = d; all_reclaims.append(out)

    legs = pd.DataFrame(all_legs)
    positions = pd.DataFrame(all_positions)
    candidates = pd.DataFrame(all_candidates)
    reclaims = pd.DataFrame(all_reclaims)
    legs.to_csv(OUT_DIR / "v15_exit_legs.csv", index=False)
    positions.to_csv(OUT_DIR / "v15_positions.csv", index=False)
    candidates.to_csv(OUT_DIR / "v15_candidate_funnel.csv", index=False)
    reclaims.to_csv(OUT_DIR / "v15_vah_reclaim_diagnostic.csv", index=False)
    return legs, positions, candidates, reclaims


try:
    _HERE = Path(__file__).resolve().parent
except NameError:
    _HERE = Path.cwd()
if not (_HERE / "data").exists():
    _HERE = _HERE.parent
all_dates = [f"2024-03-{d:02d}" for d in range(4, 29)]
agg_dir = _HERE / "data" / "aggTrades_parquet"
agg_paths = {d: agg_dir / f"{d}.parquet" for d in all_dates}

missing = [d for d in all_dates if not agg_paths[d].exists()]
if missing:
    raise FileNotFoundError(f"Missing parquet files: {missing}")

print(f"Date range: {all_dates[0]} to {all_dates[-1]} ({len(all_dates)} days)")
print(f"Test days: {all_dates[7:]} ({len(all_dates[7:])} days)")
print("Running backtest...")
legs, positions, candidates, reclaims = run_v15_backtest(
    all_dates, test_dates=all_dates[7:], agg_paths=agg_paths, book_paths=None
)
print()
print("Backtest complete!")
summary = summarize_positions(positions)
for k, v in summary.items():
    print(f"  {k}: {v}")


Date range: 2024-03-04 to 2024-03-28 (25 days)
Test days: ['2024-03-11', '2024-03-12', '2024-03-13', '2024-03-14', '2024-03-15', '2024-03-16', '2024-03-17', '2024-03-18', '2024-03-19', '2024-03-20', '2024-03-21', '2024-03-22', '2024-03-23', '2024-03-24', '2024-03-25', '2024-03-26', '2024-03-27', '2024-03-28'] (18 days)
Running backtest...
288 bars, profile val=64584
288 bars, profile val=61610
288 bars, profile val=65856
288 bars, profile val=66675
288 bars, profile val=67098
288 bars, profile val=68154
288 bars, profile val=69056
288 bars, profile val=70789
288 bars, profile val=70832
288 bars, profile val=72510
288 bars, profile val=69000
288 bars, profile val=67049
288 bars, profile val=65400
288 bars, profile val=64951
288 bars, profile val=67045
288 bars, profile val=62662
288 bars, profile val=61582
288 bars, profile val=66016
288 bars, profile val=62610
288 bars, profile val=64012
288 bars, profile val=64170
288 bars, profile val=67468
288 bars, profile val=69984
288 bars, profi

## Known limitations retained intentionally

- Sampled L1 `bookTicker` cannot support claims about stacked footprint imbalance or persistent multi-level depth.
- Five-minute OHLC cannot reveal the exact intrabar path. v15 uses conservative stop-first ordering when both stop and target are touched; tick replay should replace this assumption for final validation.
- Funding and exchange-specific fee tiers are configurable but require the actual account schedule.
- The default short and reclaim branches remain diagnostics-only until they pass standalone out-of-sample tests.
- No parameter is declared optimal from an 18-day sample. Rules must be frozen before out-of-sample evaluation.

## Recommended validation sequence

1. Run syntax and synthetic invariant tests.
2. Re-run 2024-03-11 to 2024-03-28 as an integrity comparison, not an optimization sample.
3. Reconcile every signal, entry, exit, position size, fee, and PnL manually for selected days.
4. Replay aggTrades at tick sequence for stop/target ordering.
5. Run multiple non-overlapping bull, bear, and balanced periods.
6. Use leave-one-day-out and best-trade/best-day exclusions.
7. Freeze rules and run genuine out-of-sample and paper trading.
